# NutriVisión MX: pipeline completo en Google Colab

Este notebook corre **todo el entrenamiento y la evaluación en GPU** y al final descarga un `.zip` con los resultados para copiarlos al repositorio.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU*** y luego *Ejecutar todo*.

Tiempo aproximado: **40–60 min** (la descarga de Food-101 son ~5 GB por red; el disco de Colab no se llena porque se lee como *stream*).

## 1. Verificar GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Subir el código del proyecto
Sube el archivo `nutrivision-mx-codigo.zip` cuando aparezca el botón. *(Cuando el repositorio esté en GitHub, puedes reemplazar esta celda por `!git clone <url>`.)*

In [ ]:
import os, zipfile
from google.colab import files
os.chdir("/content")
if not os.path.exists("/content/nutrivision-mx/src"):
    up = files.upload()
    name = next(iter(up))
    zipfile.ZipFile(name).extractall("/content")
os.chdir("/content/nutrivision-mx")
!ls

## 3. Dependencias
Colab ya trae PyTorch, OpenCV, scikit-learn y pandas; solo se instalan las que faltan. (Las versiones exactas usadas localmente están en `requirements.txt`.)

In [ ]:
!pip install -q imagehash onnx onnxruntime onnxscript
import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))

## 4. Datos: descarga y particiones sin fuga

In [ ]:
!python data/download.py
!python -m src.make_splits

## 5. Entrenamiento de las dos arquitecturas (mismo protocolo)

In [ ]:
!python -m src.train --arch mobilenet_v3_large --workers 2

In [ ]:
!python -m src.train --arch efficientnet_b0 --workers 2

## 6. Evaluación en prueba (una sola vez) y fuera de distribución (ISIA Food-500)

In [ ]:
!python -m src.evaluate --arch mobilenet_v3_large --split test
!python -m src.evaluate --arch efficientnet_b0 --split test
!python -m src.download_ood
!python -m src.evaluate --arch mobilenet_v3_large --split ood

## 7. Interpretabilidad, exportación a ONNX/INT8 y comparación
La latencia aquí se mide en el **CPU de Colab**; la del equipo de la demo se mide con `app/demo.py` en la laptop.

In [ ]:
!python -m src.gradcam --arch mobilenet_v3_large
!python -m src.export --arch mobilenet_v3_large --threads 2
!python -m src.export --arch efficientnet_b0 --threads 2 --f1-subset 600
!python -m src.compare

## 8. Ejecutar los notebooks de análisis (quedan guardados con sus salidas)

In [ ]:
for nb in ["notebooks/01_exploracion_datos.ipynb", "notebooks/02_curvas_entrenamiento.ipynb",
           "notebooks/03_revision_etiquetas.ipynb", "eval/robustez.ipynb",
           "eval/auditoria_subgrupos.ipynb", "eval/analisis_errores.ipynb"]:
    print("==>", nb)
    !jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=3600 "$nb" 2>&1 | grep -E "Writing|Error" 

## 9. Video de entrada para la demo

In [ ]:
!python app/make_demo_video.py --out app/demo.mp4

## 10. Empaquetar y descargar resultados
Descomprime este zip **dentro de la carpeta del proyecto** en tu computadora (sobrescribe `runs/`, `docs/`, `notebooks/`, `eval/` y agrega `app/demo.mp4`).

In [ ]:
!rm -f /content/resultados.zip
!zip -qr /content/resultados.zip runs docs notebooks eval app/demo.mp4 data/splits/reporte_particiones.json data/splits/removidas_por_fuga.csv data/ood/labels.csv -x "runs/*/export/model_pre.onnx"
!du -h /content/resultados.zip
files.download("/content/resultados.zip")